[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/04_numbering/04_numbering_lab.ipynb)


# 04 — numbering & germline (ANARCI 직접 실행)

> 본문: [`04_numbering.md`](04_numbering.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 9초** (실측)

**이 노트북은 도구를 직접 돌립니다.** ANARCI 를 **직접 돌려** IMGT·Chothia numbering CSV 를 `my_run/` 에 만들고, 커밋된 결과와 대조해요.
내 결과는 `my_run/` 에 쌓이고 커밋된 `data/` 는 대조군이에요 — 어느 단계가 실패해도 `resolve()` 가 `data/` 로 폴백해 뒤 절이 계속 돌아갑니다.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

Colab 은 이 셀 하나로 끝나고, 로컬은 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "04_numbering"
PIP_PKGS = "pandas anarci abnumber"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True    # ANARCI 계열은 hmmscan(HMMER) 실행파일이 필요해요 (pip 로는 안 깔려요)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후엔 cwd 아래만 깊이 3까지 — 위로 올라가 rglob 하면 Colab 에서 / 전체를 뒤집니다.
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
_APT = []                                # 필요한 시스템 패키지를 모아 apt 를 한 번만 돌립니다

if shutil.which("hmmscan") is None:      # ANARCI 가 부르는 실행파일 — pip 로는 안 깔려요
    if IN_COLAB:
        _APT.append("hmmer")
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if _APT:
    _run("apt-get -qq update", quiet=True)   # 인덱스가 낡으면 install 이 404 로 죽습니다
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), quiet=True)

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — ANARCI numbering (본문 4.1)

```bash
ANARCI -i data/demo_mab.fa -s imgt --assign_germline --csv --outfile my_run/demo_imgt
ANARCI -i data/demo_mab.fa -s chothia --csv --outfile my_run/demo_chothia
```
사슬별 CSV(`..._H.csv`, `..._KL.csv`)가 생깁니다. 각 numbering 위치가 **컬럼 하나**가 돼요.
`--assign_germline` 은 IMGT 쪽에만 붙였어요 — germline 은 scheme 과 무관하니 한 번만 받으면 됩니다.

In [ ]:
import pathlib, pandas as pd

run_tool(["ANARCI", "-i", "data/demo_mab.fa", "-s", "imgt",
          "--assign_germline", "--csv", "--outfile", "my_run/demo_imgt"])
run_tool(["ANARCI", "-i", "data/demo_mab.fa", "-s", "chothia",
          "--csv", "--outfile", "my_run/demo_chothia"])

# 위 출력에 뜨는 경고 두 줄은 **둘 다 무해**해요. 실패로 오해하기 쉬워 여기서 짚고 갑니다.
print("\n[경고 읽는 법] 위에 뜬 두 줄은 정상 동작 중에도 늘 나와요.")
print("  · SyntaxWarning: invalid escape sequence — ANARCI 패키지 '자기 소스' 13번째 줄에서 나요.")
print("    도움말에 그린 ASCII 아트(\\\  //)를 raw 문자열로 안 써서 Python 3.12 가 지적하는 거예요. 우리 코드와 무관해요.")
print("  · Non IG chains cannot be numbered with the chothia scheme — ANARCI 가 입력을 보기 '전에'")
print("    scheme 이 imgt 가 아니면 무조건 찍는 안내예요. 실제로 빠진 사슬이 있는지는 아래 표로 확인해요.")

# 어느 서열이 어느 파일로 갔는지 눈으로 확인 — '빠진 사슬 없음' 은 여기서 판정해요.
rows = []
for p in sorted(pathlib.Path("my_run").glob("demo_*.csv")):
    try:
        d = pd.read_csv(p)
        rows.append({"파일": p.name,
                     "scheme": "imgt" if "imgt" in p.name else "chothia",
                     "사슬": "H (중쇄)" if p.name.endswith("_H.csv") else "KL (경쇄)",
                     "번호매김된 서열": " · ".join(map(str, d["Id"])) if "Id" in d.columns else "?",
                     "numbering 컬럼": d.shape[1]})
    except Exception as e:
        rows.append({"파일": p.name, "scheme": "-", "사슬": "-",
                     "번호매김된 서열": f"읽기 실패 ({type(e).__name__})", "numbering 컬럼": 0})

if rows:
    display(pd.DataFrame(rows))
    got = {r["번호매김된 서열"] for r in rows}
    print(f"판정 — CSV {len(rows)}개(imgt/chothia × H/KL)가 정상이에요. 입력 2서열이 모두 번호매김됐으면 누락 없음이에요.")
    print("       위 표의 '번호매김된 서열' 에 demo_HC 와 demo_LC 가 다 보이면 통과예요 —", " / ".join(sorted(got)))
else:
    print("판정 — my_run 에 CSV 가 하나도 없어요. hmmscan 이 없을 가능성이 커요(Ch.03 의 3.0).")
print("       실패해도 아래 절은 커밋된 data/ 로 이어갑니다.")

## 2) 내 결과 확인 — germline 할당 (본문 4.2)

`--assign_germline` 이 붙여준 V/J gene 과 **어느 종의 germline인지**를 읽습니다.
어떤 사슬을 humanize 해야 하는지가 여기서 정해져요.

In [ ]:
import pandas as pd

FILES = {f: resolve(f) for f in ["demo_imgt_H.csv", "demo_imgt_KL.csv",
                                 "demo_chothia_H.csv", "demo_chothia_KL.csv"]}

def first_domain(fname):
    """ANARCI CSV 한 행 = variable domain 하나. 첫 domain 을 돌려주고, 여러 개면 알려준다."""
    d = pd.read_csv(FILES[fname])
    if len(d) > 1:
        print(f"[주의] {fname} 에 domain {len(d)}개 — scFv 처럼 한 서열에 VH·VL 이 같이 있으면 "
              f"행이 여러 개예요. 아래는 첫 domain 만 봅니다.")
    return d.iloc[0]

def val(r, *names):
    """있는 컬럼 중 먼저 나오는 값(없거나 NaN 이면 None)."""
    for n in names:
        if n in r.index and pd.notna(r[n]):
            return r[n]
    return None

def pct(x):
    return "-" if x is None else f"{float(x)*100:.0f}%"

germ = []
for label, f in [("Heavy", "demo_imgt_H.csv"), ("Light", "demo_imgt_KL.csv")]:
    r = first_domain(f)
    germ.append({"사슬": label,
                 "chain_type": val(r, "chain_type"),
                 "species": val(r, "identity_species", "hmm_species"),
                 "V gene": val(r, "v_gene"), "V identity": pct(val(r, "v_identity")),
                 "J gene": val(r, "j_gene"), "J identity": pct(val(r, "j_identity")),
                 "bit score": val(r, "score")})
g = pd.DataFrame(germ)
display(g)

nonhuman = [f"{row['사슬']}({row['V gene']}, {row['species']})"
            for _, row in g.iterrows() if str(row["species"]).lower() != "human"]
print("판정 — human germline 이 아닌 사슬:", ", ".join(nonhuman) or "없음")
print(f"      → {len(nonhuman)}개 사슬이 humanization 대상이에요." if nonhuman
      else "      → 두 사슬 다 human germline 이라 humanization 대상이 아니에요.")
print("      Ch.05 에서 Sapiens 도 같은 사슬만 '덜 사람스럽다'고 판정하는지 확인합니다.")
print("      bit score 는 HMM 프로파일 DB 버전에 따라 달라져요 — 5절에서 따로 봅니다(본문 4.2).")

## 3) 내 결과 확인 — IMGT vs Chothia 를 잔기 단위로 나란히 (본문 4.3)

같은 중쇄를 두 scheme 으로 numbering 하면 CDR-H1 에 들어가는 잔기 수가 달라져요.
개수만 보면 감이 안 오니 **같은 잔기가 두 scheme 에서 각각 몇 번인지** 나란히 놓고 봅니다.

In [ ]:
import pandas as pd

def numbered(fname):
    """ANARCI CSV 한 행 → [(번호 라벨, 기본 번호, 잔기)] 를 서열 순서대로."""
    r = pd.read_csv(FILES[fname]).iloc[0]
    out = []
    for col in r.index:
        lab = str(col).strip()
        base = lab.rstrip("ABCDEFGHIJKLMNOP")          # insertion code(100A) 분리
        if base.isdigit() and str(r[col]) not in ("nan", "-", ""):
            out.append((lab, int(base), str(r[col])))
    return out

def occupied(fname, lo, hi):
    return [(lab, aa) for lab, base, aa in numbered(fname) if lo <= base <= hi]

imgt_h1 = occupied("demo_imgt_H.csv", 27, 38)
chot_h1 = occupied("demo_chothia_H.csv", 26, 32)
print(f"IMGT    CDR-H1 (27-38) — {len(imgt_h1)} 잔기  {''.join(a for _, a in imgt_h1)}")
print(f"Chothia CDR-H1 (26-32) — {len(chot_h1)} 잔기  {''.join(a for _, a in chot_h1)}")

mi, mc = numbered("demo_imgt_H.csv"), numbered("demo_chothia_H.csv")
if len(mi) != len(mc) or any(a != b for (_, _, a), (_, _, b) in zip(mi, mc)):
    print("[주의] 두 CSV 의 잔기 서열이 달라 나란히 비교를 건너뜁니다 — 같은 FASTA 로 돌렸는지 확인하세요.")
else:
    lo, hi = 24, 36                                    # CDR-H1 앞뒤로 조금 넓게
    side = pd.DataFrame([
        {"서열 위치": i, "잔기": a,
         "IMGT 번호": f"H{l1}", "Chothia 번호": f"H{l2}",
         "IMGT CDR-H1": "●" if 27 <= b1 <= 38 else "",
         "Chothia CDR-H1": "●" if 26 <= b2 <= 32 else ""}
        for i, ((l1, b1, a), (l2, b2, _)) in enumerate(zip(mi, mc), 1) if lo <= i <= hi])
    display(side)

    imgt_of_chot31 = [(l1, a) for (l1, _, a), (l2, _, _) in zip(mi, mc) if l2 == "31"]
    imgt31 = [a for l, _, a in mi if l == "31"]
    print("판정 — 같은 잔기라도 scheme 이 다르면 번호가 달라져요.")
    for l1, a in imgt_of_chot31:
        print(f"      Chothia H31 = {a}  ↔  같은 잔기의 IMGT 번호는 H{l1}")
    print("      IMGT H31 은", f"{imgt31[0]}" if imgt31 else "이 서열에서 비어 있어요(gap)",
          "— 그래서 scheme 없는 'H31' 은 어느 잔기인지 알 수 없어요.")
    print("      보고서·mutation table 에는 항상 (IMGT) 또는 (Chothia) 를 명시하세요.")

## 4) 내 결과 확인 — 서열 QC 표 (본문 4.4)

numbering 은 QC 8단계 중 2~4단계예요. 지금까지 읽은 값을 한 장으로 모아 두면
Ch.05 이후 모든 챕터가 이 표를 좌표계로 씁니다.

In [ ]:
import pandas as pd

IMGT_CDR = {"CDR1 (IMGT)": (27, 38), "CDR2 (IMGT)": (56, 65), "CDR3 (IMGT)": (105, 117)}

fa, name = {}, None
for line in open("data/demo_mab.fa"):
    line = line.strip()
    if line.startswith(">"): name = line[1:].split()[0]; fa[name] = ""
    elif name: fa[name] += line

def qc_col(fname):
    r = pd.read_csv(FILES[fname]).iloc[0]
    n = len(numbered(fname))
    sid = str(val(r, "Id"))
    col = {"Chain type": val(r, "chain_type"),
           "V gene": f"{val(r, 'v_gene')} ({val(r, 'identity_species', 'hmm_species')}, {pct(val(r, 'v_identity'))})",
           "J gene": f"{val(r, 'j_gene')} ({pct(val(r, 'j_identity'))})"}
    for lab, (lo, hi) in IMGT_CDR.items():
        s = "".join(a for _, a in occupied(fname, lo, hi))
        col[lab] = f"{s} ({len(s)} aa)"
    col["Numbering 성공"] = "양호" if n else "실패"
    col["Numbering 잔기 수"] = n
    col["Sequence length"] = len(fa.get(sid, ""))
    return col

qc = pd.DataFrame({"Heavy": qc_col("demo_imgt_H.csv"), "Light": qc_col("demo_imgt_KL.csv")})
display(qc)

covered = True
for lab in qc.columns:
    n, L = qc.loc["Numbering 잔기 수", lab], qc.loc["Sequence length", lab]
    covered &= (n == L)
    print(f"      {lab}: numbering {n} / FASTA {L} 잔기 →",
          "전체를 덮었어요." if n == L else f"{L - n} 잔기가 variable domain 밖이에요(constant 영역 등).")
print("판정 —", "두 사슬 다 numbering 이 서열 전체를 덮었어요. QC 통과예요."
      if covered else "덮이지 않은 구간이 있어요 — FASTA 에 constant 영역이 섞였는지 확인하세요.")
print("      QC 8단계 중 여기까지가 1~4단계. 5~6(liability)은 Ch.08, 7(reference 비교)은 Ch.09, 8(구조)은 Ch.06 이에요.")

## 5) 레퍼런스 대조 — 커밋된 ANARCI 결과와 같은가 (본문 4.2)

`data/demo_imgt_H.csv` 등은 이 저장소를 만들 때 ANARCI 로 돌려 커밋해 둔 대조군이에요.
비교는 두 갈래로 나눠서 봅니다 — **germline·numbering**(같아야 하는 것)과
**bit score·e-value**(도구/HMM 프로파일 DB 버전에 따라 달라질 수 있는 것).

In [ ]:
import pandas as pd, pathlib

SOFT = ["score", "e-value"]          # 버전에 따라 달라질 수 있는 값 (본문 4.2)

def fmt(vals):
    """대괄호·따옴표·부동소수 찌꺼기를 걷어내요 — 2.6000000000000003e-60 → 2.6e-60."""
    out = []
    for v in vals:
        try:
            x = float(v)
            out.append(f"{x:g}" if 1e-4 <= abs(x) < 1e5 else f"{x:.2e}")
        except (TypeError, ValueError):
            out.append(str(v))
    return " · ".join(out)

rows, hard_diffs = [], []
for fname in ["demo_imgt_H.csv", "demo_imgt_KL.csv",
              "demo_chothia_H.csv", "demo_chothia_KL.csv"]:
    mine_p, ref_p = pathlib.Path("my_run")/fname, pathlib.Path("data")/fname
    if not mine_p.exists() or not ref_p.exists():
        rows.append({"파일": fname,
                     "numbering 일치": "대조 건너뜀",
                     "대조한 컬럼": "-",
                     "bit score (내 결과 → 레퍼런스)": "my_run 산출물 없음" if not mine_p.exists() else "레퍼런스 없음",
                     "e-value (내 결과 → 레퍼런스)": "-"})
        continue
    mine, ref = pd.read_csv(mine_p), pd.read_csv(ref_p)
    common = [c for c in ref.columns if c in mine.columns]
    hard   = [c for c in common if c not in SOFT]          # 같아야 하는 것
    same   = mine[hard].astype(str).equals(ref[hard].astype(str))
    row = {"파일": fname,
           "numbering 일치": "O 같음" if same else "X 다름",
           "대조한 컬럼": f"{len(hard)} / 공통 {len(common)}"}
    for col in SOFT:                                        # 달라도 되는 것
        m, rf = (fmt(mine[col].tolist()), fmt(ref[col].tolist())) if col in common else ("-", "-")
        row[f"{col} (내 결과 → 레퍼런스)"] = m if m == rf else f"{m} → {rf}"
    rows.append(row)
    if not same:                                            # 진짜 문제일 때만 자세히
        for col in hard:
            if not mine[col].astype(str).equals(ref[col].astype(str)):
                hard_diffs.append((fname, col, fmt(mine[col].tolist()), fmt(ref[col].tolist())))

display(pd.DataFrame(rows))

if hard_diffs:
    print("germline·numbering 이 어긋난 자리 — 여기가 진짜 문제예요")
    for fname, col, m, rf in hard_diffs:
        print(f"  {fname} · {col} — 내 결과 {m} / 레퍼런스 {rf}")

judged = [r for r in rows if r["numbering 일치"] != "대조 건너뜀"]
ok = sum(1 for r in judged if r["numbering 일치"] == "O 같음")
if not judged:
    print("대조할 my_run 산출물이 없어요 — 1절 셀을 먼저 실행하세요.")
else:
    print(f"판정 — 대조한 {len(judged)}개 파일 중 germline·numbering 일치 {ok}개.")
    print("       위 표에서 'numbering 일치' 가 전부 O 면 통과예요.")
    print("       bit score·e-value 칸에 화살표(→)가 보여도 정상이에요 — ANARCI/HMM 프로파일 DB")
    print("       버전이 다르면 점수만 조금 달라지고 numbering 은 그대로거든요. 보고서엔 도구 버전을 함께 적으세요.")

> 다음 → 본문 [05. humanness & humanization](../05_humanness/05_humanness.md)